# CORD LoRA 微调 · Qwen3.5-4B (A100 / bf16)

**用法**：
1. 顶部菜单 `修改 → 笔记本设置 → 硬件加速器 → A100 GPU`
2. `代码执行程序 → 全部运行`
3. 运行到「上传数据」cell 时，选本地 `train.train.jsonl` 和 `val.train.jsonl`
4. 训完会自动打包 `cord_adapter.zip` 让你下载 → 解压到本地 `adapters/cord/`

产出：HF PEFT 格式 adapter（vLLM 多 LoRA 可直接加载）。


## 1. 装依赖


In [ ]:
!pip install -q unsloth

## 2. 上传训练数据（选 train.train.jsonl 和 val.train.jsonl）


In [ ]:
from google.colab import files
print("请选择 train.train.jsonl 和 val.train.jsonl ...")
up = files.upload()
print("已上传:", list(up.keys()))

## 3. 配置（超参对齐实验计划）


In [ ]:
BASE        = "unsloth/Qwen3.5-4B"
MAX_SEQ_LEN = 2048
LOAD_IN_4BIT= False          # A100 用 bf16；若只有 T4 改 True
RANK, ALPHA, DROPOUT = 16, 32, 0.05
EPOCHS, LR  = 3, 2e-4
BATCH, GRAD_ACCUM = 2, 8     # 有效 batch = 16
TRAIN_FILE, VAL_FILE = "train.train.jsonl", "val.train.jsonl"
OUT = "cord_adapter"

## 4. 载入基座（Qwen3.5 → FastModel）


In [ ]:
from unsloth import FastModel
model, tokenizer = FastModel.from_pretrained(
    model_name   = BASE,
    max_seq_length = MAX_SEQ_LEN,
    load_in_4bit = LOAD_IN_4BIT,
    full_finetuning = False,
)

## 5. 套 LoRA（r=16，目标全部 linear）


In [ ]:
model = FastModel.get_peft_model(
    model,
    r = RANK, lora_alpha = ALPHA, lora_dropout = DROPOUT,
    target_modules = ["q_proj","k_proj","v_proj","o_proj",
                      "gate_proj","up_proj","down_proj"],
    use_gradient_checkpointing = "unsloth",
    random_state = 42,
)

## 6. 数据：messages → chat 文本


In [ ]:
from datasets import load_dataset
def fmt(ex):
    return {"text": tokenizer.apply_chat_template(ex["messages"], tokenize=False)}
train_ds = load_dataset("json", data_files=TRAIN_FILE, split="train").map(fmt)
eval_ds  = load_dataset("json", data_files=VAL_FILE,   split="train").map(fmt)
print("train =", len(train_ds), " eval =", len(eval_ds))
print("---- sample ----\n", train_ds[0]["text"][:400])

## 7. Trainer（只在 assistant 回答上算 loss）


In [ ]:
from trl import SFTConfig, SFTTrainer
trainer = SFTTrainer(
    model = model, tokenizer = tokenizer,
    train_dataset = train_ds, eval_dataset = eval_ds,
    args = SFTConfig(
        dataset_text_field = "text",
        per_device_train_batch_size = BATCH,
        gradient_accumulation_steps = GRAD_ACCUM,
        num_train_epochs = EPOCHS,
        learning_rate = LR,
        lr_scheduler_type = "cosine",
        warmup_ratio = 0.05,
        optim = "adamw_8bit",
        max_seq_length = MAX_SEQ_LEN,
        logging_steps = 10,
        save_strategy = "epoch",
        eval_strategy = "epoch",
        output_dir = "outputs",
        seed = 42,
        report_to = "none",
    ),
)
from unsloth.chat_templates import train_on_responses_only
trainer = train_on_responses_only(
    trainer,
    instruction_part = "<|im_start|>user\n",
    response_part    = "<|im_start|>assistant\n",
)
print("trainer ready")

## 8. 训练


In [ ]:
trainer_stats = trainer.train()

## 9. 保存 adapter + 打包下载


In [ ]:
model.save_pretrained(OUT)
tokenizer.save_pretrained(OUT)
!ls -la cord_adapter

import shutil
shutil.make_archive("cord_adapter", "zip", OUT)
from google.colab import files
files.download("cord_adapter.zip")

## 10. 快速 sanity 推理（看一条 val 样本输出）


In [ ]:
import json, torch
FastModel.for_inference(model)
ex = json.loads(open(VAL_FILE).readline())
msgs = ex["messages"][:2]   # system + user
ids = tokenizer.apply_chat_template(msgs, tokenize=True, add_generation_prompt=True, return_tensors="pt").to("cuda")
out = model.generate(input_ids=ids, max_new_tokens=512, do_sample=False)
print("=== 模型输出 ===")
print(tokenizer.decode(out[0][ids.shape[1]:], skip_special_tokens=True))
print("\n=== GOLD ===")
print(ex["messages"][2]["content"])